# Student modelling sandbox

This is a **standalone, editable synthetic exercise**. Its ten entities,
continuous response, and binary `hit` target are invented. They are not
molecules, assays, QSAR evidence, or biological evidence.

**Question.** How do fitting, validation-based selection, and held-out
error analysis fit together without allowing test labels to choose a model?
You need basic Python plus the idea of train/validation/test partitions.

**Colab upload.** Download this file, open
[Google Colab](https://colab.research.google.com/), choose **File → Upload
notebook**, and select it. CPU is enough; there is no package installation,
API key, network request, GPU, or Drive mount. Run cells from top to bottom.
The report is written to a fresh temporary folder in the current runtime;
download it yourself if you want to retain it.

## 1. Imports and a fresh output folder

These are Python standard-library modules and the `IPython.display` helper
already included with Jupyter and Colab. `mkdtemp` creates a new directory
beneath the current working directory. The notebook does not replace an
earlier report or alter author result files.

In [ ]:
import json
import math
import tempfile
from pathlib import Path

from IPython.display import HTML, SVG, display

OUTPUT_DIR = Path(tempfile.mkdtemp(prefix="student-modelling-", dir=Path.cwd()))
print("Fresh student output:", OUTPUT_DIR)

## 2. Inspect the constructed records before fitting

Each `entity_id` is unique. The scalar `feature` is deliberately not a
molecular descriptor. The last two records form the test partition: one is
extrapolation and one contradicts the training line. Change a **copy** only
after you predict which quantities should change.

In [ ]:
def teaching_rows():
    """Authored numbers, not molecules, assay measurements, or a benchmark."""
    values = [
        (0, 1, 0, "train"),
        (1, 3, 0, "train"),
        (2, 5, 0, "train"),
        (3, 7, 1, "train"),
        (4, 9, 1, "train"),
        (5, 11, 1, "train"),
        (1.5, 4, 0, "validation"),
        (3.5, 8, 1, "validation"),
        (6, 10, 1, "test"),
        (7, 9, 0, "test"),
    ]
    return [
        {"entity_id": f"toy-{i + 1:02}", "feature": x, "response": y, "hit": hit, "split": split}
        for i, (x, y, hit, split) in enumerate(values)
    ]

rows = teaching_rows()
rows

## 3. Exact fitting and scoring mechanics

Read these short pure functions before the protocol cell. Ridge minimizes
sum of squared residuals plus `alpha * slope²`; its intercept is free.
Logistic fitting uses 500 fixed full-batch steps on mean cross-entropy plus
`alpha * slope² / 2`, after training-only standardization. A probability is
clipped only while calculating log loss. These are instructional mechanics,
not an assay model or a convergence guarantee.

In [ ]:
def mean(values):
    return sum(values) / len(values)

def fit_ridge(x, y, alpha):
    """Minimize sum squared error + alpha * slope squared; intercept is free."""
    x_bar, y_bar = mean(x), mean(y)
    denominator = sum((value - x_bar) ** 2 for value in x) + alpha
    slope = (
        sum((a - x_bar) * (b - y_bar) for a, b in zip(x, y, strict=True)) / denominator
        if denominator
        else 0.0
    )
    return y_bar - slope * x_bar, slope

def sigmoid(value):
    if value >= 0:
        return 1 / (1 + math.exp(-value))
    exponential = math.exp(value)
    return exponential / (1 + exponential)

def fit_logistic(x, y, alpha, steps=500, start=(0.0, 0.0)):
    """Fixed full-batch steps on mean CE + alpha * slope squared / 2.

    The step budget and the starting point are arguments so that the optimiser
    itself can be varied without touching the data, the penalty or the selection
    rule. Defaults reproduce the original fit exactly.
    """
    center = mean(x)
    scale = math.sqrt(mean([(value - center) ** 2 for value in x])) or 1.0
    z = [(value - center) / scale for value in x]
    intercept, slope = start
    for _ in range(steps):
        residuals = [
            sigmoid(intercept + slope * value) - target for value, target in zip(z, y, strict=True)
        ]
        intercept -= 0.1 * mean(residuals)
        slope -= 0.1 * (mean([a * b for a, b in zip(residuals, z, strict=True)]) + alpha * slope)
    return {"intercept": intercept, "slope": slope, "mean": center, "scale": scale}

def predict_logistic(model, x):
    return [
        sigmoid(model["intercept"] + model["slope"] * (value - model["mean"]) / model["scale"])
        for value in x
    ]

def log_loss(y, probabilities):
    clipped = [min(max(value, 1e-15), 1 - 1e-15) for value in probabilities]
    return -mean(
        [
            target * math.log(p) + (1 - target) * math.log1p(-p)
            for target, p in zip(y, clipped, strict=True)
        ]
    )

def classification_metrics(y, probabilities):
    pairs = list(zip(y, [p >= 0.5 for p in probabilities], strict=True))
    tp = sum(target == 1 and prediction for target, prediction in pairs)
    tn = sum(target == 0 and not prediction for target, prediction in pairs)
    fp = sum(target == 0 and prediction for target, prediction in pairs)
    fn = sum(target == 1 and not prediction for target, prediction in pairs)
    return {
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "accuracy": (tp + tn) / len(y),
        "precision": tp / (tp + fp) if tp + fp else None,
        "recall": tp / (tp + fn) if tp + fn else None,
    }

def regression_metrics(y, predictions):
    errors = [prediction - target for target, prediction in zip(y, predictions, strict=True)]
    return {
        "mae": mean([abs(value) for value in errors]),
        "rmse": math.sqrt(mean([value**2 for value in errors])),
    }

def fit_stump(x, y, min_leaf):
    """Pick the training split that minimises squared error, both sides at least min_leaf.

    This is the smallest member of the tree family: one threshold, two constants.
    It shares nothing with the linear family except the records it is given, which
    is what makes the comparison a comparison of assumptions rather than of tuning.
    """
    order = sorted(zip(x, y, strict=True))
    thresholds = sorted({(a[0] + b[0]) / 2 for a, b in zip(order, order[1:], strict=False)})
    best = None
    for threshold in thresholds:
        left = [value for feature, value in order if feature < threshold]
        right = [value for feature, value in order if feature >= threshold]
        if len(left) < min_leaf or len(right) < min_leaf:
            continue
        left_mean, right_mean = mean(left), mean(right)
        error = sum((value - left_mean) ** 2 for value in left) + sum(
            (value - right_mean) ** 2 for value in right
        )
        if best is None or error < best[0]:
            best = (error, threshold, left_mean, right_mean)
    if best is None:
        # No admissible split: the family degenerates to the constant it contains.
        return None, mean(y), mean(y)
    return best[1], best[2], best[3]

def predict_stump(model, x):
    threshold, left, right = model
    if threshold is None:
        return [left] * len(x)
    return [left if value < threshold else right for value in x]

## 4. Validate and partition before fitting

This is the first visible fragment of the canonical protocol, not a call to
a runner. It rejects duplicate IDs, invalid splits and invalid labels before
extracting training and validation values. No test target is read here.

In [ ]:
"""Split first. Fit train only; select on validation; score test only afterward."""
if not rows or len({row["entity_id"] for row in rows}) != len(rows):
    raise ValueError("Entity IDs must be unique across all partitions")
partitions = {name: [] for name in ("train", "validation", "test")}
for row in rows:
    if row["split"] not in partitions:
        raise ValueError("Unknown partition")
    if not all(math.isfinite(row[key]) for key in ("feature", "response", "hit")):
        raise ValueError("Features and labels must be finite")
    if row["hit"] not in (0, 1):
        raise ValueError("The constructed hit target must be binary")
    partitions[row["split"]].append(row)
if any(not values for values in partitions.values()):
    raise ValueError("Each partition must contain records")
train, validation, test = (partitions[name] for name in ("train", "validation", "test"))
x = [row["feature"] for row in train]
y = [row["response"] for row in train]
hits = [row["hit"] for row in train]
validation_x = [row["feature"] for row in validation]

print({name: len(values) for name, values in partitions.items()})

## 5. Fit candidate models using training records only

Two families are fitted here on the same training records. The first is
linear: ridge for the response and logistic for the constructed hit, over
three penalties. The second is the smallest member of the tree family, a
stump that picks one threshold and predicts a constant on each side, over
three minimum leaf sizes.

The penalty and the leaf size play the same role. Each one constrains how
much the family may bend to the training records. What differs between the
families is not their tuning but their assumption about the shape of the
relationship, which is the choice this section exists to expose. Neither
loop inspects a test label.

In [ ]:
candidates, validation_scores = {}, []
for alpha in (0.0, 1.0, 10.0):
    regression = fit_ridge(x, y, alpha)
    classification = fit_logistic(x, hits, alpha)
    candidates[str(alpha)] = {"regression": regression, "classification": classification}
    predictions = [regression[0] + regression[1] * value for value in validation_x]
    validation_scores.append(
        {
            "alpha": alpha,
            "regression_mae": regression_metrics(
                [row["response"] for row in validation],
                predictions,
            )["mae"],
            "classification_log_loss": log_loss(
                [row["hit"] for row in validation],
                predict_logistic(classification, validation_x),
            ),
        }
    )
stump_candidates, stump_validation_scores = {}, []
for min_leaf in (1, 2, 3):
    stump = fit_stump(x, y, min_leaf)
    stump_candidates[str(min_leaf)] = stump
    stump_validation_scores.append(
        {
            "min_leaf": min_leaf,
            "regression_mae": regression_metrics(
                [row["response"] for row in validation],
                predict_stump(stump, validation_x),
            )["mae"],
            "split_threshold": stump[0],
        }
    )

## 6. Inspect validation scores, then select

Run and inspect this cell before the next one. It selects lower validation
MAE for regression and lower validation log loss for classification; exact
ties use lower alpha. The same rule then selects the stump's leaf size, so
the two families are chosen under one procedure and differ only in what
they assume. The displayed table is deliberately available before any test
scoring.

Read the stump table before moving on. All three leaf sizes produce the
same threshold and the same validation MAE here, so the grid decided
nothing and the tie rule picked the least constrained candidate. A grid
that cannot separate its candidates is a fact about this dataset, not a
reason to widen the grid until something wins.

In [ ]:
# Tuple ordering fixes ties in favor of the lower alpha; no test target is used here.
regression_alpha = min(
    validation_scores, key=lambda row: (row["regression_mae"], row["alpha"])
)["alpha"]
classification_alpha = min(
    validation_scores, key=lambda row: (row["classification_log_loss"], row["alpha"])
)["alpha"]
regression = candidates[str(regression_alpha)]["regression"]
classification = candidates[str(classification_alpha)]["classification"]
# The same rule, applied to the other family: lowest validation MAE, ties to the
# less constrained candidate. Only the assumption set differs.
stump_min_leaf = min(
    stump_validation_scores, key=lambda row: (row["regression_mae"], row["min_leaf"])
)["min_leaf"]
stump = stump_candidates[str(stump_min_leaf)]

display(HTML('<pre>' + json.dumps(validation_scores, indent=2) + '</pre>'))

## 7. Measure what the optimiser alone contributes

The configuration is now fixed. This cell changes nothing about the data,
the penalty or the selection rule, and varies only the search: three
declared starting points crossed with three step budgets. There is no
randomness, so every difference below comes from the optimisation and not
from a seed.

Read the budget table first. Compute is counted as gradient evaluations,
the step budget times the six training records, and the two questions asked
of each budget are whether the three starting points agree on the
**decision** at the 0.5 threshold, and how far apart their probabilities
still are.

The spread across starting points is not a confidence interval. It measures
how far the search has travelled, not how much the six records constrain
the answer. A well-converged optimiser with badly insufficient data would
show a small spread and still be uncertain.

In [ ]:
# With the configuration chosen, vary only the optimiser. Three declared
# starting points and three step budgets, no randomness and no new data, so
# every difference below is the search and not the records.
optimisation_variability = []
for steps in (50, 200, 500):
    for start in ((0.0, 0.0), (-2.0, 1.0), (2.0, -1.0)):
        attempt = fit_logistic(x, hits, classification_alpha, steps=steps, start=start)
        validation_probabilities = predict_logistic(attempt, validation_x)
        optimisation_variability.append(
            {
                "steps": steps,
                "start": list(start),
                "gradient_evaluations": steps * len(train),
                "train_log_loss": log_loss(hits, predict_logistic(attempt, x)),
                "validation_probabilities": validation_probabilities,
                "validation_decisions": [
                    int(probability >= 0.5) for probability in validation_probabilities
                ],
            }
        )
budget_summary = []
for steps in (50, 200, 500):
    attempts = [row for row in optimisation_variability if row["steps"] == steps]
    decisions = {tuple(row["validation_decisions"]) for row in attempts}
    spreads = [
        max(column) - min(column)
        for column in zip(*(row["validation_probabilities"] for row in attempts), strict=True)
    ]
    budget_summary.append(
        {
            "steps": steps,
            "gradient_evaluations": attempts[0]["gradient_evaluations"],
            "starting_points": len(attempts),
            "decisions_agree": len(decisions) == 1,
            "largest_probability_spread": max(spreads),
        }
    )

display(HTML('<pre>' + json.dumps(budget_summary, indent=2) + '</pre>'))

## 8. Score the held-out records once

Only after selection does this final canonical fragment read the test
records, produce predictions and calculate errors. It does not refit with
validation rows. Read the report and avoid using these values to change the
already selected recipe.

In [ ]:
test_x = [row["feature"] for row in test]
predictions = [regression[0] + regression[1] * value for value in test_x]
stump_predictions = predict_stump(stump, test_x)
probabilities = predict_logistic(classification, test_x)
targets = [row["response"] for row in test]
test_hits = [row["hit"] for row in test]
student_report = {
    "data_kind": "synthetic-teaching-example",
    "claim": "Training mechanics only; not QSAR performance or biological evidence",
    "counts": {name: len(values) for name, values in partitions.items()},
    "selection": {
        "regression_alpha": regression_alpha,
        "stump_min_leaf": stump_min_leaf,
        "classification_alpha": classification_alpha,
        "classification_threshold": 0.5,
        "refit_with_validation": False,
    },
    "validation_scores": validation_scores,
    "stump_validation_scores": stump_validation_scores,
    "optimisation_variability": optimisation_variability,
    "budget_summary": budget_summary,
    "candidate_models": candidates,
    "models": {
        "regression": regression,
        "classification": classification,
        "stump": stump,
        "train_mean": mean(y),
        "train_prevalence": mean(hits),
    },
    "test_metrics": {
        "ridge": regression_metrics(targets, predictions),
        "stump": regression_metrics(targets, stump_predictions),
        "train_mean": regression_metrics(targets, [mean(y)] * len(test)),
        "logistic": {
            **classification_metrics(test_hits, probabilities),
            "log_loss": log_loss(test_hits, probabilities),
        },
        "train_prevalence": {
            **classification_metrics(test_hits, [mean(hits)] * len(test)),
            "log_loss": log_loss(test_hits, [mean(hits)] * len(test)),
        },
    },
    "test_predictions": [
        {
            **row,
            "prediction": prediction,
            "residual": prediction - row["response"],
            "hit_probability": probability,
        }
        for row, prediction, probability in zip(test, predictions, probabilities, strict=True)
    ],
}

display(HTML('<pre>' + json.dumps(student_report, indent=2) + '</pre>'))

## 9. Plot what was actually fitted

The SVG below is made directly from the selected models and the current
records. Circles show continuous responses by partition; the dark line is
the selected ridge prediction. The response axis includes the whole drawn
extrapolation. Bars show selected logistic probabilities and a 0.5 reference.
This is a small visual check, not a claim of model quality.

In [ ]:
width, height, left, bottom = 820, 430, 70, 330
features = [row["feature"] for row in rows]
responses = [row["response"] for row in rows]
x_min, x_max = min(features) - 0.5, max(features) + 0.5
intercept, slope = student_report["models"]["regression"]
displayed_predictions = [intercept + slope * value for value in (x_min, x_max)]
y_min = math.floor(min(*responses, *displayed_predictions) - 1)
y_max = math.ceil(max(*responses, *displayed_predictions) + 1)
def x_to_px(value):
    return left + (value - x_min) / (x_max - x_min) * 290

def y_to_px(value):
    return bottom - (value - y_min) / (y_max - y_min) * 250
colors = {"train": "#2563eb", "validation": "#d97706", "test": "#dc2626"}
line = " ".join(
    f"{x_to_px(value):.1f},{y_to_px(intercept + slope * value):.1f}"
    for value in (x_min, x_max)
)
circles = "".join(
    f'<circle cx="{x_to_px(row["feature"]):.1f}" cy="{y_to_px(row["response"]):.1f}" r="5" fill="{colors[row["split"]]}"/>'
    for row in rows
)
probabilities = predict_logistic(
    student_report["models"]["classification"], features
)
probability_left, probability_top, probability_bottom = 470, 80, bottom
def probability_to_px(value):
    return probability_bottom - value * (probability_bottom - probability_top)
bars = "".join(
    f'<rect x="{probability_left + 20 + index * 25}" y="{probability_to_px(probability):.1f}" width="15" height="{probability_bottom - probability_to_px(probability):.1f}" fill="#7c3aed"/>'
    for index, probability in enumerate(probabilities)
)
response_ticks = "".join(
    f'<line x1="{left - 4}" y1="{y_to_px(value):.1f}" x2="{left}" y2="{y_to_px(value):.1f}" stroke="black"/><text x="{left - 9}" y="{y_to_px(value) + 4:.1f}" text-anchor="end" font-size="11">{value}</text>'
    for value in range(y_min, y_max + 1, 2)
)
feature_ticks = "".join(
    f'<text x="{x_to_px(value):.1f}" y="{bottom + 18}" text-anchor="middle" font-size="11">{value}</text>'
    for value in range(math.ceil(x_min), math.floor(x_max) + 1)
)
probability_ticks = "".join(
    f'<text x="{probability_left - 9}" y="{probability_to_px(value) + 4:.1f}" text-anchor="end" font-size="11">{value:.1f}</text>'
    for value in (0.0, 0.5, 1.0)
)
entity_labels = "".join(
    f'<text x="{probability_left + 28 + index * 25}" y="{bottom + 18}" text-anchor="end" font-size="9" transform="rotate(-55 {probability_left + 28 + index * 25} {bottom + 18})">{row["entity_id"]}</text>'
    for index, row in enumerate(rows)
)
svg = f'''<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">
<rect width="100%" height="100%" fill="white"/><text x="70" y="25" font-size="15">Ridge response (left) and logistic probability (right)</text>
<line x1="{left}" y1="{bottom}" x2="{left + 290}" y2="{bottom}" stroke="black"/><line x1="{left}" y1="{probability_top}" x2="{left}" y2="{bottom}" stroke="black"/>{response_ticks}{feature_ticks}<polyline points="{line}" fill="none" stroke="#111827" stroke-width="2"/>{circles}
<text x="{left + 145}" y="{bottom + 40}" text-anchor="middle" font-size="12">synthetic feature</text><text x="18" y="205" font-size="12" transform="rotate(-90 18 205)">constructed response</text>
<circle cx="80" cy="55" r="5" fill="#2563eb"/><text x="90" y="59" font-size="11">train</text><circle cx="145" cy="55" r="5" fill="#d97706"/><text x="155" y="59" font-size="11">validation</text><circle cx="245" cy="55" r="5" fill="#dc2626"/><text x="255" y="59" font-size="11">test</text><line x1="305" y1="55" x2="325" y2="55" stroke="#111827" stroke-width="2"/><text x="332" y="59" font-size="11">selected ridge</text>
<line x1="{probability_left}" y1="{bottom}" x2="760" y2="{bottom}" stroke="black"/><line x1="{probability_left}" y1="{probability_top}" x2="{probability_left}" y2="{bottom}" stroke="black"/>{probability_ticks}<line x1="{probability_left}" y1="{probability_to_px(0.5):.1f}" x2="760" y2="{probability_to_px(0.5):.1f}" stroke="#6b7280" stroke-dasharray="4 3"/><text x="615" y="65" text-anchor="middle" font-size="11">decision threshold = 0.5</text>{bars}{entity_labels}
<text x="615" y="{bottom + 72}" text-anchor="middle" font-size="12">constructed entity ID</text><text x="428" y="205" font-size="12" transform="rotate(-90 428 205)">hit probability (0 to 1)</text></svg>'''
svg_path = OUTPUT_DIR / "fitted_models.svg"
with svg_path.open("x", encoding="utf-8") as handle:
    handle.write(svg)
display(SVG(svg))

## 10. Read the constructed predictive failure against baselines

Lower MAE and log loss are better. On these intentionally contradictory
held-out labels, selected ridge has a worse MAE than the training-mean
baseline. Logistic has the same thresholded accuracy as prevalence, but a
much worse log loss because it assigns a very confident probability to the
negative test entity. These are observed toy outputs, not evidence about
real molecular prediction.

The stump reaches a much lower test MAE than either the linear model or the
constant. Read why before reading it as a win. The test features lie beyond
every training feature, and the constructed responses turn over there. A
linear model must extrapolate, so a slope fitted on the rising training
records carries it far past the true values. A stump cannot extrapolate at
all: outside the training range it repeats a leaf constant, and here that
refusal happens to land close to the held-out responses.

That is an argument about assumptions and a data regime, not about tree
methods being stronger. Two test entities cannot establish which family is
better, and a stump that had landed on the wrong side of its threshold
would have looked equally decisive in the other direction.

In [ ]:
ridge_mae = student_report["test_metrics"]["ridge"]["mae"]
stump_mae = student_report["test_metrics"]["stump"]["mae"]
mean_mae = student_report["test_metrics"]["train_mean"]["mae"]
logistic_loss = student_report["test_metrics"]["logistic"]["log_loss"]
prevalence_loss = student_report["test_metrics"]["train_prevalence"]["log_loss"]
threshold, left, right = student_report["models"]["stump"]
display(HTML(
    f"<p><b>Observed constructed comparison:</b> ridge MAE = {ridge_mae:.3f} "
    f"versus stump MAE = {stump_mae:.3f} versus training-mean MAE = {mean_mae:.3f}; "
    f"logistic log loss = {logistic_loss:.3f} versus prevalence log loss = "
    f"{prevalence_loss:.3f}.</p>"
    f"<p>The selected stump splits at {threshold} and predicts {left} below it and "
    f"{right} at or above it, so every test feature receives {right} no matter how "
    f"far outside the training range it lies. That is the assumption, not a fit.</p>"
))

## 11. Save your observed synthetic result

The exclusive-create mode (`"x"`) would reject an existing target. Here the
parent was freshly made in step 1, so a re-run from the top makes a separate
report folder. The JSON is a record of this constructed exercise only.

In [ ]:
report_path = OUTPUT_DIR / "student_report.json"
with report_path.open("x", encoding="utf-8") as handle:
    json.dump(student_report, handle, indent=2, allow_nan=False)
    handle.write("\n")
print("Saved:", report_path)

## 12. Independent exercises, then answers

1. Add 100 to both **test** responses. Which candidates, selections, and
   predictions must stay unchanged? Which output changes?
2. Replace both validation responses with 6. Which fitted candidates stay
   unchanged, and which regression alpha should selection favor?
3. Duplicate an entity ID. Should fitting start?
4. Why can thresholded accuracy match while logistic and prevalence log loss
   differ strongly?
5. The stump beat both the linear model and the constant on these test
   records. Name the property of the two families that produced the
   difference, and say what would have to change about the **records**, not
   the models, for the linear family to win instead.
6. Going from 50 to 200 steps costs 900 further gradient evaluations and
   changes the decision agreement. Going from 200 to 500 costs 1800 more
   and does not. What should that pattern change about how you report the
   result, and why is the remaining probability spread still not an
   uncertainty estimate?

**Answers.** (1) Candidate parameters, selection, and predictions remain
fixed; residuals and regression test errors change. (2) Candidates stay
fixed; alpha 10 is closest to 6. (3) Validation rejects duplicate IDs before
fitting. (4) Accuracy discards confidence; a confident wrong probability is
heavily penalized by log loss. (5) The linear family extrapolates and the
stump does not: beyond the training range the stump repeats a leaf constant
while the fitted slope keeps rising. The constructed responses turn over
outside the training features, so refusing to extrapolate lands closer. Put
the test features inside the training range, or make the response continue
rising through them, and the ordering reverses. The families were selected
under one procedure, so the difference is the assumption, not the tuning.
(6) The cheaper budget already fixes the decision, so the extra compute buys
agreement in a quantity nobody acts on. Report the budget at which the
decision stopped moving and state it as part of the recipe, rather than
quoting the largest budget as though it were required. The remaining spread
is a distance between three searches on identical data, so it can shrink to
nothing while the six records still fail to determine the answer. An
estimate of that would have to come from the data, not from the optimiser.
None of these toy outcomes supports a claim about unseen molecules or
biology, and two test entities cannot rank model families.